In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def resolve_hw_dir() -> Path:
    # Чтобы ноутбук работал независимо от того, где стартовал Jupyter.
    candidates = [
        Path.cwd(),
        Path.cwd() / "HW08-09",
        Path.cwd() / "homeworks" / "HW08-09",
        Path.cwd() / "makaroni_po_flotski" / "homeworks" / "HW08-09",
    ]
    for c in candidates:
        if (c / "HW08-09.ipynb").exists() or (c / "practic").exists():
            return c
    return Path.cwd()


hw_dir = resolve_hw_dir()
data_root = hw_dir / "data"
artifacts_dir = hw_dir / "artifacts"
fig_dir = artifacts_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Датасет: EMNIST(split="balanced").
# Если нет доступа к интернету/датасет недоступен, можно включить синтетический fallback,
# чтобы проверить пайплайн (НО для сдачи нужен реальный датасет!).
ALLOW_SYNTHETIC_FALLBACK = False

transform = transforms.ToTensor()
dataset_name = "EMNIST(balanced)"
num_classes = 47


def load_emnist(train: bool):
    try:
        return datasets.EMNIST(
            root=str(data_root),
            split="balanced",
            train=train,
            download=False,
            transform=transform,
        )
    except Exception:
        return datasets.EMNIST(
            root=str(data_root),
            split="balanced",
            train=train,
            download=True,
            transform=transform,
        )


try:
    train_full = load_emnist(train=True)
    test_ds = load_emnist(train=False)
except Exception as e:
    if not ALLOW_SYNTHETIC_FALLBACK:
        raise RuntimeError(
            "Failed to load/download EMNIST (split='balanced'). "
            "Fix internet access or ensure the dataset exists in ./data. "
            "(download=False is tried first; then download=True.)"
        ) from e

    print("WARNING: failed to load/download EMNIST. Falling back to synthetic data.")
    print("ERROR:", repr(e))
    dataset_name = "SYNTHETIC"
    num_classes = 47

    n_total = 6000
    x = torch.rand(n_total, 1, 28, 28)
    y = torch.randint(0, num_classes, (n_total,))
    train_full = TensorDataset(x, y)

    x_test = torch.rand(1000, 1, 28, 28)
    y_test = torch.randint(0, num_classes, (1000,))
    test_ds = TensorDataset(x_test, y_test)


val_ratio = 0.2
n_train = int((1 - val_ratio) * len(train_full))
n_val = len(train_full) - n_train

train_ds, val_ds = random_split(
    train_full,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

batch_size = 128

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))
input_dim = int(np.prod(x_batch.shape[1:]))
print("Batch:", x_batch.shape, y_batch.shape)
print("Dataset:", dataset_name, "| input_dim:", input_dim, "| num_classes:", num_classes)


class MLP(nn.Module):
    def __init__(self, hidden_sizes=(256, 256), dropout_p=0.0, use_batchnorm=False):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            in_dim = h
        layers.append(nn.Linear(in_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)


def accuracy_from_logits(logits, targets):
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, total_count = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        total_count += bs

    return total_loss / total_count, total_acc / total_count


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, total_count = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy_from_logits(logits, y) * bs
        total_count += bs

    return total_loss / total_count, total_acc / total_count


def run_experiment(
    experiment_id,
    hidden_sizes=(256, 256),
    dropout_p=0.0,
    use_batchnorm=False,
    optimizer_name="Adam",
    lr=1e-3,
    momentum=0.0,
    weight_decay=0.0,
    max_epochs=20,
    early_stopping_patience=None,
    early_stopping_min_delta=0.0,
):
    model = MLP(hidden_sizes, dropout_p, use_batchnorm).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        momentum_to_log = 0.0
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
        momentum_to_log = float(momentum)
    else:
        raise ValueError("optimizer_name must be 'Adam' or 'SGD'")

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(1, max_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"[{experiment_id}] Epoch {epoch}: val_acc={val_acc:.4f}")

        if val_acc > best_val_acc + early_stopping_min_delta:
            best_val_acc = float(val_acc)
            best_val_loss = float(val_loss)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if early_stopping_patience and patience_counter >= early_stopping_patience:
            print(f"[{experiment_id}] Early stopping!")
            break

    epochs_trained = len(history["val_acc"])
    model_summary = f"hidden={hidden_sizes} | act=ReLU | dropout={dropout_p} | bn={use_batchnorm}"

    return {
        "experiment_id": experiment_id,
        "history": history,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "best_val_loss": best_val_loss,
        "best_state_dict": best_state,
        "model_summary": model_summary,
        "config": {
            "hidden_sizes": hidden_sizes,
            "dropout_p": dropout_p,
            "use_batchnorm": use_batchnorm,
            "optimizer": optimizer_name,
            "lr": lr,
            "momentum": momentum_to_log,
            "weight_decay": weight_decay,
            "dataset": dataset_name,
            "seed": SEED,
        },
    }


# ---------------- EXPERIMENTS E1–E4 ----------------

runs = []

E1 = run_experiment("E1", hidden_sizes=(256, 256), dropout_p=0.0, use_batchnorm=False, max_epochs=20)
runs.append(E1)

E2 = run_experiment("E2", hidden_sizes=(256, 256), dropout_p=0.3, use_batchnorm=False, max_epochs=20)
runs.append(E2)

E3 = run_experiment("E3", hidden_sizes=(256, 256), dropout_p=0.0, use_batchnorm=True, max_epochs=20)
runs.append(E3)

best_reg = max([E2, E3], key=lambda r: r["best_val_accuracy"])
print("Best regularization run:", best_reg["experiment_id"], "val_acc=", best_reg["best_val_accuracy"])

cfg = best_reg["config"]
E4 = run_experiment(
    "E4",
    hidden_sizes=cfg["hidden_sizes"],
    dropout_p=cfg["dropout_p"],
    use_batchnorm=cfg["use_batchnorm"],
    optimizer_name=cfg["optimizer"],
    lr=cfg["lr"],
    max_epochs=50,
    early_stopping_patience=4,
    early_stopping_min_delta=0.0005,
)
runs.append(E4)

best_overall = E4
best_cfg = best_overall["config"]


# ---------------- EXPERIMENTS O1–O3 ----------------

O1 = run_experiment(
    "O1",
    hidden_sizes=best_cfg["hidden_sizes"],
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
    optimizer_name="Adam",
    lr=1e-1,
    max_epochs=6,
)
runs.append(O1)

O2 = run_experiment(
    "O2",
    hidden_sizes=best_cfg["hidden_sizes"],
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
    optimizer_name="Adam",
    lr=1e-5,
    max_epochs=6,
)
runs.append(O2)

O3 = run_experiment(
    "O3",
    hidden_sizes=best_cfg["hidden_sizes"],
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
    optimizer_name="SGD",
    lr=1e-2,
    momentum=0.9,
    weight_decay=1e-4,
    max_epochs=15,
)
runs.append(O3)


# ---------------- SAVE ARTIFACTS ----------------

rows = []
for r in runs:
    cfg = r["config"]
    rows.append(
        {
            "experiment_id": r["experiment_id"],
            "dataset": cfg["dataset"],
            "seed": cfg["seed"],
            "model_summary": r["model_summary"],
            "optimizer": cfg["optimizer"],
            "lr": cfg["lr"],
            "momentum": cfg["momentum"],
            "weight_decay": cfg["weight_decay"],
            "epochs_trained": r["epochs_trained"],
            "best_val_accuracy": r["best_val_accuracy"],
            "best_val_loss": r["best_val_loss"],
        }
    )

df = pd.DataFrame(rows)
df.to_csv(artifacts_dir / "runs.csv", index=False)
df

torch.save(best_overall["best_state_dict"], artifacts_dir / "best_model.pt")


# ---------------- PLOTS ----------------

hist = best_overall["history"]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist["train_loss"], label="train_loss")
plt.plot(hist["val_loss"], label="val_loss")
plt.title("E4: loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(hist["train_acc"], label="train_acc")
plt.plot(hist["val_acc"], label="val_acc")
plt.title("E4: accuracy")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig(fig_dir / "curves_best.png")
plt.close()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(O1["history"]["val_loss"], label="O1 (lr=1e-1)")
plt.plot(O2["history"]["val_loss"], label="O2 (lr=1e-5)")
plt.title("LR extremes: val_loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(O1["history"]["val_acc"], label="O1 (lr=1e-1)")
plt.plot(O2["history"]["val_acc"], label="O2 (lr=1e-5)")
plt.title("LR extremes: val_acc")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig(fig_dir / "curves_lr_extremes.png")
plt.close()


# ---------------- FINAL TEST EVAL (E4, once) ----------------

best_model = MLP(
    hidden_sizes=best_cfg["hidden_sizes"],
    dropout_p=best_cfg["dropout_p"],
    use_batchnorm=best_cfg["use_batchnorm"],
).to(device)
best_model.load_state_dict(best_overall["best_state_dict"])

criterion = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
print("Final test accuracy:", test_acc)

best_config = dict(best_overall["config"])
best_config.update(
    {
        "epochs_trained": best_overall["epochs_trained"],
        "best_val_accuracy": best_overall["best_val_accuracy"],
        "best_val_loss": best_overall["best_val_loss"],
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
    }
)

with open(artifacts_dir / "best_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)
